# Projeto IHTC-2024: Otimização NRA (Nurse-to-Room Assignment)

Este *notebook* executa a bateria de testes e gera as análises visuais e estatísticas para comparar as três abordagens desenvolvidas: **Heurística Gulosa**, **Algoritmo Genético (POO)** e **Solver PLI (Matheurística)**.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.experiments import run_suite, greedy_baseline
from src.solver import pli_solver
from src.genetic_algorithm import ga_solver

# Configurações de visualização para o Notebook
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

## 1. Execução da Bateria de Testes
Aqui definimos as instâncias de teste e executamos o motor de otimização (`run_suite`) garantindo o rigor estatístico com $R=5$ repetições independentes.

In [ ]:
print("Iniciando bateria de testes IHTC-2024...")

base_dir = Path.cwd()
data_dir = base_dir / "data"
out_dir = base_dir / "results"
out_dir.mkdir(exist_ok=True)

methods_to_test = {
    "Greedy Baseline": greedy_baseline,
    "Genetic Algorithm": ga_solver,
    "Solver PLI": pli_solver
}

# executa a suíte
df_results = run_suite(
    data_dir = data_dir,
    instance_ids = ["i04", "i06"],
    methods = methods_to_test,
    repeats = 5, 
    base_seed = 42,
    out_dir = out_dir
)

print("Testes concluídos com sucesso!")

## 2. Tabela Estatística de Resultados
Resumo com Mínimo, Média e Desvio Padrão (`std`) para validar a estabilidade e performance das abordagens.

In [ ]:
resumo = df_results.groupby(["instance_id", "method"])["objective"].agg(["min", "mean", "std"]).round(2)
display(resumo)

## 3. Análise Comparativa: Qualidade da Solução vs Tempo
Visualização lado a lado da distribuição dos custos e do custo computacional de cada algoritmo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# boxplot de Custos
sns.boxplot(data=df_results, x="instance_id", y="objective", hue="method", palette="Set2", ax=axes[0])
axes[0].set_title("Distribuição do Custo (Penalidades) por Instância")
axes[0].set_ylabel("Custo Total")
axes[0].set_xlabel("Instância")

# gráfico de Barras de Tempo de Execução 
sns.barplot(data=df_results, x="instance_id", y="runtime_s", hue="method", palette="pastel", errorbar=None, ax=axes[1])
axes[1].set_title("Tempo Médio de Execução")
axes[1].set_ylabel("Tempo (Segundos)")
axes[1].set_xlabel("Instância")

plt.tight_layout()
plt.show()

## 4. Comportamento da Meta-heurística (Convergência)
Análise da descida do gradiente estocástico para observar a fase de exploração nas múltiplas repetições.

In [ ]:
history_dir = Path("results/convergence")
files = list(history_dir.glob("hist_i06_*_mut0.15.json"))

if files:
    plt.figure(figsize=(10, 5))
    for file in files:
        with open(file, 'r') as f:
            history = json.load(f)
            plt.plot(history, alpha=0.6, linewidth=2)

    plt.title("Convergência do Algoritmo Genético (5 Seeds Independentes) - Instância i06")
    plt.xlabel("Gerações")
    plt.ylabel("Custo Objetivo")
    plt.show()
else:
    print("Arquivos de histórico não encontrados. Rode a bateria principal primeiro.")

## 5. Análise de Sensibilidade (Taxa de Mutação)
Executa de maneira isolada o Algoritmo Genético variando um hiperparâmetro para discutir o impacto de mínimos locais.

In [ ]:
from src.data_loader import load_instance

print("A processar testes de sensibilidade...")
mutations = [0.01, 0.15, 0.50]
plt.figure(figsize=(10, 5))

inst = load_instance(data_dir=data_dir, instance_id="i04")

for mut in mutations:
    ga_solver(inst, seed=42, mutation_rate=mut)
    
    file = Path(f"results/convergence/hist_i04_seed42_mut{mut}.json")
    if file.exists():
        with open(file, 'r') as f:
            history = json.load(f)
            plt.plot(history, label=f"Taxa de Mutação = {mut*100}%", linewidth=2.5)
            
plt.title("Análise de Sensibilidade: Impacto da Mutação na Convergência (Instância i04)")
plt.xlabel("Gerações")
plt.ylabel("Custo Objetivo")
plt.legend()
plt.show()